# Grad-CAM Smoke Test

Use this notebook after the Grad-CAM branch is merged. It restores the run #4 checkpoint from Hugging Face, regenerates TBX11K JSONL locally, runs Grad-CAM on one validation sample per class, and displays the original image, heatmap, overlay, and metadata inline.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ShivamSinghNow/Drishti.git"
BRANCH = "main"
WORKDIR = Path("/content/Drishti")

HF_REPO_ID = "ShivSingh123/drishti-qlora-run4-vision-lora-ablation"
CHECKPOINT_REPO_PATH = "checkpoints/checkpoint-4950"
ADAPTER_DIR = Path("outputs/dri19-run4-vision-lora-ablation/checkpoint-4950")
GRADCAM_OUTPUT_DIR = Path("outputs/gradcam/run4-checkpoint-4950-smoke")

SAMPLE_LABELS = ("active_tb", "healthy", "sick_but_non_tb")
TARGET_LABEL = "predicted"
LAYER_INDEX = -1
OVERLAY_ALPHA = 0.45

In [ ]:
if not WORKDIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

%cd {WORKDIR}
!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q -r requirements-colab.txt

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

def require_secret(name: str) -> str:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Missing Colab secret: {name}")
    return value

os.environ["KAGGLE_USERNAME"] = require_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = require_secret("KAGGLE_KEY")
os.environ["HF_TOKEN"] = require_secret("HF_TOKEN")

login(token=os.environ["HF_TOKEN"])
api = HfApi(token=os.environ["HF_TOKEN"])
print(api.repo_info(repo_id=HF_REPO_ID, repo_type="model").id)

In [ ]:
import importlib.metadata as metadata
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU before continuing.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"gpu={gpu_name} vram_gb={gpu_mem_gb:.1f}")
print(f"torch={torch.__version__} cuda={torch.version.cuda}")
for package in ("transformers", "peft", "bitsandbytes", "accelerate", "huggingface_hub"):
    print(f"{package}={metadata.version(package)}")

In [ ]:
!python download_dataset.py
!python generate_jsonl.py --output-dir data/processed

import json
from collections import Counter

expected = {
    "train": Counter({"active_tb": 600, "healthy": 3000, "sick_but_non_tb": 3000}),
    "val": Counter({"active_tb": 200, "healthy": 800, "sick_but_non_tb": 800}),
}

for split, expected_counts in expected.items():
    counts = Counter()
    with open(f"data/processed/{split}.jsonl", encoding="utf-8") as handle:
        for line in handle:
            payload = json.loads(line)
            assistant = payload["messages"][1]["content"]
            assert assistant.startswith("Classification: "), assistant
            assert assistant.count("\n") == 0, assistant
            counts[assistant.removeprefix("Classification: ")] += 1
    print(split, dict(sorted(counts.items())))
    assert counts == expected_counts, (split, counts, expected_counts)

In [ ]:
from huggingface_hub import snapshot_download
import shutil

snapshot_dir = snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="model",
    allow_patterns=[f"{CHECKPOINT_REPO_PATH}/*"],
)

downloaded = Path(snapshot_dir) / CHECKPOINT_REPO_PATH
ADAPTER_DIR.parent.mkdir(parents=True, exist_ok=True)
if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)
shutil.copytree(downloaded, ADAPTER_DIR, symlinks=False)

print(ADAPTER_DIR.exists(), ADAPTER_DIR)
print(sorted(p.name for p in ADAPTER_DIR.iterdir()))

In [ ]:
def extract_label(payload: dict) -> str:
    assistant = payload["messages"][1]["content"]
    return assistant.removeprefix("Classification: ").strip()

sample_indices = {}
with open("data/processed/val.jsonl", encoding="utf-8") as handle:
    for index, line in enumerate(handle):
        label = extract_label(json.loads(line))
        if label in SAMPLE_LABELS and label not in sample_indices:
            sample_indices[label] = index
        if len(sample_indices) == len(SAMPLE_LABELS):
            break

print(sample_indices)
missing = set(SAMPLE_LABELS).difference(sample_indices)
if missing:
    raise RuntimeError(f"Could not find val samples for: {sorted(missing)}")

In [ ]:
import subprocess

def run(command: list[str]) -> None:
    print("\n$ " + " ".join(command))
    subprocess.run(command, check=True)

for label, sample_index in sample_indices.items():
    output_dir = GRADCAM_OUTPUT_DIR / label
    run([
        "python", "generate_gradcam.py",
        "--adapter-dir", str(ADAPTER_DIR),
        "--data-dir", "data/processed",
        "--split", "val",
        "--sample-index", str(sample_index),
        "--target-label", TARGET_LABEL,
        "--layer-index", str(LAYER_INDEX),
        "--overlay-alpha", str(OVERLAY_ALPHA),
        "--output-dir", str(output_dir),
    ])

In [ ]:
from IPython.display import display
from PIL import Image

for label in SAMPLE_LABELS:
    output_dir = GRADCAM_OUTPUT_DIR / label
    metadata_files = sorted(output_dir.glob("*_metadata.json"))
    if not metadata_files:
        print(f"No metadata found for {label}")
        continue
    metadata = json.loads(metadata_files[0].read_text(encoding="utf-8"))
    print("\n===", label, "===")
    print(json.dumps({
        "sample_index": metadata["sample_index"],
        "true_label": metadata["true_label"],
        "predicted_label": metadata["predicted_label"],
        "target_label": metadata["target_label"],
        "cam_method": metadata["cam_method"],
        "hook_layer_name": metadata["hook_layer_name"],
        "spatial_shape": metadata["spatial_shape"],
        "class_probabilities": metadata["class_probabilities"],
    }, indent=2))
    display(Image.open(metadata["image_path"]).resize((384, 384)))
    display(Image.open(metadata["output_heatmap"]))
    display(Image.open(metadata["output_overlay"]))

## Optional: Try A Different Layer Or Target

If the last-block map is too diffuse, rerun the Grad-CAM command with `LAYER_INDEX = -2` or `-3`. If you want to explain a counterfactual class rather than the model's prediction, set `TARGET_LABEL` to `active_tb`, `healthy`, or `sick_but_non_tb` and rerun the Grad-CAM/display cells.